# 01 Tensor 与 Autograd 基础

## 本节目标

- 理解 Tensor 在 PyTorch 中如何表示数据、参数和中间结果。
- 观察 shape、广播和矩阵乘法这些基础操作。
- 从代码角度理解 `requires_grad`、`backward()` 和 `.grad` 的关系。
- 用一个很小的优化例子感受自动求导如何服务训练循环。

## 背景问题

本节关注的问题是：当模型输出和目标之间存在差距时，PyTorch 如何知道参数应该往哪个方向调整？

Tensor 负责承载数据和参数，Autograd 负责记录计算过程并计算梯度。这个概念可以从代码角度理解为：只要一个 Tensor 开启了 `requires_grad=True`，后续可求导运算就会被 PyTorch 跟踪。

## 核心概念

- Tensor：多维数组，可以表示输入样本、模型参数、梯度和中间激活。
- Shape：张量每个维度的大小，初学者容易在这里混淆 batch 维和特征维。
- Broadcasting：形状兼容时，PyTorch 会自动扩展较小的 Tensor 参与计算。
- Autograd：根据 Tensor 运算构建计算图，并在 `backward()` 时计算梯度。

## 最小代码示例

下面先从最小 Tensor 操作开始。调试时可以优先检查每一步的 shape，因为很多训练错误不是模型复杂，而是维度没有对齐。

In [ ]:
import torch
import numpy as np

torch.manual_seed(42)

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.ones(2)
print("a shape:", a.shape)
print("b shape:", b.shape)
print("a + b =")
print(a + b)  # b 会按行广播

In [ ]:
x = torch.randn(3, 4)
w = torch.randn(4, 2)
y = x @ w
print("x:", x.shape)
print("w:", w.shape)
print("y:", y.shape)

### 自动求导小例子

这里用一个单变量函数观察梯度。这里的关键不是公式本身，而是确认 PyTorch 计算出的梯度和我们对函数变化方向的理解一致。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x
y.backward()
print("y:", y.item())
print("dy/dx:", x.grad.item())

## 完整实验

下面把自动求导放进一个最小训练闭环：预测、loss、反向传播、参数更新。这个实验用于观察梯度方向是否能推动参数向更合理的值移动。

In [ ]:
w = torch.tensor(0.5, requires_grad=True)
target = torch.tensor(6.0)
lr = 0.1

for step in range(8):
    prediction = 4 * w
    loss = (prediction - target) ** 2
    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad

    w.grad.zero_()
    print(f"step={step:02d}, w={w.item():.4f}, loss={loss.item():.4f}")

## 实验观察

从运行结果可以看到，`w` 会逐步靠近能让 `4 * w = 6` 的位置，loss 也会随之下降。这说明在这个简单问题上，梯度方向符合预期。

## 常见错误

- 对没有开启 `requires_grad` 的 Tensor 期待 `.grad`，结果发现没有梯度。
- 多次调用 `backward()` 但不清空梯度，导致梯度累积。
- 在还需要求导的地方使用 `.detach()`，切断了计算图。

初学者容易在这里混淆“Tensor 有数值”和“Tensor 会记录梯度”这两件事。

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = x * 2
y.backward()
print("first grad:", x.grad.item())

y = x * 3
y.backward()
print("accumulated grad:", x.grad.item())

x.grad.zero_()
print("after zero:", x.grad.item())

## 概念问答

**Q：为什么 PyTorch 默认会累积梯度？**  
A：这可以支持梯度累积等训练方式。普通训练循环中通常每个 batch 前都要清空梯度。

**Q：`.item()` 有什么用？**  
A：它把只有一个元素的 Tensor 转成 Python 数值，常用于打印 loss。

**Q：什么时候需要 `torch.no_grad()`？**  
A：当只是更新参数、评估模型或做推理时，不需要构建新的计算图，可以使用 `torch.no_grad()`。

## 小结

Tensor 是 PyTorch 计算的基础，Autograd 让梯度计算自动化。掌握 shape、梯度累积和 `backward()`，后续写训练循环会顺很多。